In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
# Q1. How many lesson pages
print(len(documents))
# Answer: 72

72


### Q2. Indexing and searching
Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

What's the filename of the first result?

- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/14-agentic-loop.md
- 04-evaluation/lessons/13-llm-as-judge.md
- 06-best-practices/lessons/02-hybrid-search.md

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

def search(content):
    return index.search(
        content
    )

query="How does the agentic loop keep calling the model until it stops?"
results = search(query)
print(results[0]['filename'])

# 01-agentic-rag/lessons/14-agentic-loop.md

01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

question = "How does the agentic loop keep calling the model until it stops?"

result = assistant.rag(question)

print(result.answer)
print(result.usage.input_tokens) # 10320

It keeps calling the model inside a `while True` loop.

Each turn it:

1. sends the current `messages` to the model,
2. checks the response for any `function_call` items,
3. runs those tool calls and appends the outputs to `messages`,
4. sets `has_function_calls = True` if any tool was called.

At the end of the turn, it breaks only if `has_function_calls == False`, meaning the model returned a final message with no more tool calls.
10320


In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

# How many chinks do we get?
print(len(chunks)) # 295

295


In [9]:
# Build the chunk index
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

In [10]:
# Create a search function that uses the chunk index
def search(query: str) -> list:
    """
    Search the course lesson pages for content matching the given query.
    """
    return chunk_index.search(query, num_results=5)

In [14]:
# Build an agent with your search tool and run it, with toyaikit

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

instructions = """You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering."""

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [15]:
# Count how many times search was called

from openai.types.responses.response_function_tool_call import ResponseFunctionToolCall

search_calls = [
    m for m in result.all_messages
    if isinstance(m, ResponseFunctionToolCall) and m.name == "search"
]
print(f"Search called {len(search_calls)} times")  # → 3

Search called 4 times
